# Ujian Tengah Semester - Graf Sains Data

**Nama :** Bima Setia Sugiharto

**NIM :** 24031554040

**Kelas :** 2024A

# 1. Tugas Pendahuluan
## 1.1 Persiapan dan Perkenalan Lingkungan
Pembuatan virtual environment sangat disarankan untuk proses isolasi *development environment* sehingga dapat meminimalisasi "disrupsi" paket-paket dependensi dari *local projects* di lingkungan kerja (laptop / PC) masing-masing.

### Langkah-langkah di Terminal:
1. **Membuat virtual environment baru**:
   ```bash
   python -m venv myenv
   ```
2. **Menonaktifkan paket virtual environment lain (jika ada) dan mengaktifkan virtual environment baru**:
   * Windows PowerShell: `.\myenv\Scripts\Activate.ps1`
   * Unix / macOS: `source myvenv/bin/activate`
3. **Menginstal paket-paket dependensi dari file `requirements.txt`**:
   ```bash
   pip install -r requirements.txt
   ```

In [1]:
# 1.1.3 Verifikasi Pustaka Dependensi yang Terpasang di Virtual Environment
import sys
import neo4j
import pandas as pd
import tqdm

print(f"Versi Python        : {sys.version.split()[0]}")
print(f"Driver neo4j version: {neo4j.__version__}")
print(f"Pandas version      : {pd.__version__}")
print(f"Tqdm version        : {tqdm.__version__}")
print("Status: Lingkungan virtual & dependensi siap digunakan!")

Versi Python        : 3.12.5
Driver neo4j version: 6.4.1
Pandas version      : 3.0.6
Tqdm version        : 4.70.1
Status: Lingkungan virtual & dependensi siap digunakan!


## 1.2 Mempelajari Penggunaan Driver Neo4J
Pada tahap ini, kita menguji koneksi driver ke database Neo4j lokal, melakukan operasi penambahan node `:Person` beserta relasi `[:KNOWS]`, membaca data grafik, serta menjawab pertanyaan pendahuluan.

In [ ]:
# 1.2.2 mendefinisikan string koneksi ke database Neo4J
from neo4j import GraphDatabase, RoutingControl

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "Bimabelajargraf")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print(f"Berhasil terhubung ke database Neo4J di {URI}!")
    
    driver.execute_query("MATCH (n) DETACH DELETE n;", database_="neo4j")
    print("Total Reset berhasil: Konten database neo4j telah dibersihkan.")

Berhasil terhubung ke database Neo4J di bolt://localhost:7687!
Total Reset berhasil: Konten database neo4j telah dibersihkan.


In [ ]:
# 1.2.3 pembuatan sebuah Node dengan relationship-nya
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_="neo4j",
    )

# 1.2.4 Untuk mencetak dan menjalankan keseluruhan
def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

In [ ]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    print("Menambahkan relasi pertemanan Arthur...")
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    
    print("\nDaftar teman Arthur:")
    print_friends(driver, "Arthur")

Menambahkan relasi pertemanan Arthur...

Daftar teman Arthur:
Guinevere
Lancelot
Merlin


### Pertanyaan Pendahuluan (Checkpoint 1.2)

#### 1. Apa nama database yang digunakan pada tutorial ini?
> **Jawaban:** Database yang digunakan adalah **`neo4j`**, sebagaimana didefinisikan pada parameter `database_="neo4j"` dalam pemanggilan fungsi `driver.execute_query()`.

#### 2. Sebutkan akses lain (seperti sintaks DDL/DML CYPHER) yang dapat digunakan mengakses database lain dalam 1 instance!
> **Jawaban:**
> * **Klausul Cypher** `USE` yang digunakan di awal query untuk menentukan target database yang dituju, contoh:  
>   ```cypher
>   USE system SHOW DATABASES;
>   USE movie MATCH (m:Movie) RETURN count(m);
>   ```
> * **Sintaks DDL Cypher** yang digunakan untuk administrasi *multi-database*:  
>   ```cypher
>   SHOW DATABASES;
>   CREATE DATABASE mynewdb;
>   STOP DATABASE mynewdb;
>   START DATABASE mynewdb;
>   DROP DATABASE mynewdb;
>   ```
> * **Driver Session / API Parameter** untuk menentukan target database saat membuka session atau eksekusi kueri pada Python driver, yakni: 
>   ```python
>   session = driver.session(database="nama_db")
>   driver.execute_query(query, database_="nama_db")
>   ```

#### 3. Apa jenis graf pertemanan yang telah dibuat di sini?
> **Jawaban:**
> * **Graf Berarah (*Directed Graph*)**. Karena relasi memiliki arah orientasi eksplisit dari sumber ke tujuan `(a)-[:KNOWS]->(friend)`.
> * **Graf Homogen / Monopartit (*Homogeneous / Unipartite Graph*)**. Karena semua simpul memiliki jenis/label entitas yang sama, yaitu entitas `:Person`.
> * **Property Graph Model**. Karena entitas berupa simpul berlabel `:Person` dengan properti `name`, dan tepi berlabel relasi `:KNOWS`.

## 1.3 Melakukan Proses Impor Data dari RetailRocket ke Database Neo4J
* **Sumber data:** Roman Zykov (RetailRocket Recommender System Dataset)
* **Perintah Download Kaggle (Windows PowerShell)**:
  ```powershell
  curl.exe -L -o ecommerce-dataset.zip `
    https://www.kaggle.com/api/v1/datasets/download/retailrocket/ecommerce-dataset
  ```
* **Rincian Fitur Data (`events.csv`)**:
  * `timestamp`: Waktu dalam format unix epoch (milidetik).
  * `visitorid`: ID unik pengunjung (Node `:Visitor`).
  * `event`: Jenis interaksi default: `view`, `addtocart`, `transaction`.
  * `itemid`: ID unik produk (Node `:Item`).
  * `transactionid`: ID transaksi, hanya terisi saat event berjenis `transaction`.

In [ ]:
# 1.3.3 Konfigurasi Argumen & Pemeriksaan Berkas events.csv
import os
import sys
import argparse
import pandas as pd
from neo4j import GraphDatabase
from tqdm import tqdm

def parse_arguments():
    parser = argparse.ArgumentParser(
        description="Impor dataset RetailRocket (events.csv) ke database Neo4J."
    )
    parser.add_argument("-u", "--user", default="neo4j", help="Username Neo4j (default: neo4j)")
    parser.add_argument("-p", "--password", default="Bimabelajargraf", help="Password Neo4J (default: Bimabelajargraf)")
    parser.add_argument("-b", "--bolt", default="bolt://127.0.0.1:7687", help="Neo4J URI / Bolt (default: bolt://127.0.0.1:7687)")
    parser.add_argument("-s", "--source", default="data", help="Direktori/folder sumber data (default: data)")
    parser.add_argument("--batch-size", type=int, default=1000, help="Batch size (default: 1000)")
    parser.add_argument("--limit", type=int, default=5000, help="Limit baris (default: 5000 untuk pengujian)")
    return parser.parse_args(args=[])

args = parse_arguments()

csv_file_path = os.path.join(args.source, "events.csv")

if not os.path.exists(csv_file_path):
    print(f"Error: File '{csv_file_path}' tidak ditemukan!")
    sys.exit(1)
else:
    print(f"File dataset valid: {csv_file_path} ditemukan.")

try:
    driver = GraphDatabase.driver(
        args.bolt, auth=(args.user, args.password)
    )
    driver.verify_connectivity()
    print("Koneksi ke Neo4j berhasil diverifikasi.")
except Exception as e:
    print(f"Gagal terhubung ke database: {e}")
    sys.exit(1)

File dataset valid: data\events.csv ditemukan.
Koneksi ke Neo4j berhasil diverifikasi.


In [ ]:
# 1.3.4 & 1.3.5 Pembuatan Constraint dan Query Cypher Batch Import

def create_constraints(session):
    session.run("CREATE CONSTRAINT unique_visitor IF NOT EXISTS FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE;")
    session.run("CREATE CONSTRAINT unique_item IF NOT EXISTS FOR (i:Item) REQUIRE i.itemId IS UNIQUE;")

def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})

    WITH v, i, row
    CALL {
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row
        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

In [ ]:
# 1.3.4 Eksekusi Proses Impor dengan Mekanisme Batch
batch_size = args.batch_size

with driver.session() as session:
    create_constraints(session)

    print(f"Membaca file data dari: {csv_file_path}")
    if args.limit:
        total_rows = args.limit
        chunks = pd.read_csv(csv_file_path, chunksize=batch_size, nrows=total_rows)
    else:
        with open(csv_file_path, "rb") as f:
            total_rows = sum(1 for _ in f) - 1
        chunks = pd.read_csv(csv_file_path, chunksize=batch_size)

    total_chunks = (total_rows // batch_size) + (1 if total_rows % batch_size != 0 else 0)

    print("Memulai proses impor data ke Neo4j...")
    for chunk in tqdm(chunks, total=total_chunks, desc="Mengimpor Batch Data"):
        chunk["transactionid"] = (
            chunk["transactionid"].fillna("").astype(str)
        )

        records = chunk.to_dict(orient="records")
        session.execute_write(import_batch, records)

    print("Proses impor berhasil.")

Membuat constraint unik untuk Visitor dan Item...
Membaca file data dari: data\events.csv
Memulai proses impor data ke Neo4j...
Mengimpor Batch Data: 100%|██████████| 5/5 [00:02<00:00,  1.85it/s]
Proses impor berhasil.


In [ ]:
with driver.session() as session:
    stat_query = """
    MATCH (v:Visitor) WITH count(v) as visitors
    MATCH (i:Item) WITH visitors, count(i) as items
    MATCH ()-[rv:VIEWED]->() WITH visitors, items, count(rv) as views
    MATCH ()-[ra:ADDED_TO_CART]->() WITH visitors, items, views, count(ra) as carts
    MATCH ()-[rb:BOUGHT]->() WITH visitors, items, views, carts, count(rb) as buys
    RETURN visitors, items, views, carts, buys
    """
    res = session.run(stat_query).single()
    
    print("=" * 50)
    print("  STATISTIK GRAF RETAILROCKET DI DATABASE NEO4J")
    print("=" * 50)
    print(f"Total Simpul Pengunjung (:Visitor) : {res['visitors']:,}")
    print(f"Total Simpul Produk     (:Item)    : {res['items']:,}")
    print(f"Total Relasi Lihat      [:VIEWED]  : {res['views']:,}")
    print(f"Total Relasi Keranjang  [:ADDED]   : {res['carts']:,}")
    print(f"Total Relasi Pembelian  [:BOUGHT]  : {res['buys']:,}")
    print("-" * 50)
    print(f"Total Keseluruhan Interaksi Edge   : {res['views'] + res['carts'] + res['buys']:,}")
    print("=" * 50)

driver.close()

  STATISTIK GRAF RETAILROCKET DI DATABASE NEO4J
Total Simpul Pengunjung (:Visitor) : 3,862
Total Simpul Produk     (:Item)    : 4,031
Total Relasi Lihat      [:VIEWED]  : 4,836
Total Relasi Keranjang  [:ADDED]   : 123
Total Relasi Pembelian  [:BOUGHT]  : 41
--------------------------------------------------
Total Keseluruhan Interaksi Edge   : 5,000
